In [2]:
import arxiv
from arxiv import Search, SortCriterion, Client
from typing import List,Dict,Any,Optional
from pydantic import BaseModel,Field
from datetime import datetime
import logging
import itertools
from dataclasses import dataclass

In [3]:
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger()

In [4]:
@dataclass
class PaperMetadata:
    paper_id: str
    title: str
    authors: List[str]
    abstract: str
    url: str
    published_date: str

In [18]:
def search_arxiv(query: str, max_results: int = 1) -> List[PaperMetadata]:
    """ search arxiv and return papers metadata"""
    logging.info(f"searching arxiv papers for : {query}")
    search = arxiv.Search(
            query = query,
            max_results = max_results,
            sort_by = arxiv.SortCriterion.Relevance
    )
    
    client = Client()
    
    papers = []
    for result in client.results(search):
        paper = PaperMetadata(
            paper_id = result.entry_id.split('/')[-1] ,
            title = result.title,
            authors = [a.name for a in result.authors],
            abstract = result.summary,
            url = result.pdf_url,
            published_date = result.published.strftime("%Y-%m-%d")
        )
        papers.append(paper)
    logging.info(f"found {len(papers)} papers")
    return papers

In [19]:
import requests
import PyPDF2
import io
def extract_text_from_pdf(pdf_path: str) -> str:
    """ download pdf and extract text """
    logger.info(f"fetching pdf from url :{url}")
    #download
    response=requests.get(url)
    pdf_file = io.BytesIO(response.content)
    #extract text
    reader = PyPDF2.PdfReader(pdf_file)
    text =""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text 

In [20]:
def chunk_text(text: str, paper_id: str, 
                     chunk_size: int = 1000, overlap: int = 200) ->  List[str]:
        """Chunk text and generate embeddings"""
        try:
            logger.info(f" Chunking text for paper: {paper_id}")
            
            # Chunk text
            chunks = self._chunk_text(text, paper_id, chunk_size, overlap)
            
            logger.info(f" Created {len(chunks)} chunks")
            
        except Exception as e:
            logger.error(f" Error: {e}")
           

In [21]:
summary_instructions = """
You're a helpful assistant that helps user by summarizing the research article by using tools like search_arxiv to retrieve papers related to query and
use tools like extract_text_from_pdf to get text from research pdf

"""

In [25]:
from agents import Agent, function_tool, Runner
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIAgentsSDKRunner

chat_interface = IPythonChatInterface()

In [26]:
from toyaikit.tools import wrap_instance_methods
summarize_tools = wrap_instance_methods(search_arxiv,extract_text_from_pdf)

In [27]:
summarize_agent = Agent(
    name='summarize_agent',
    instructions = summary_instructions,
    model='gpt-4o-mini',
    tools=summarize_tools
)

In [28]:
runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=summarize_agent
)

await runner.run();

You: attention is all you need


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


You: stop


Chat ended.
